In [2]:
import json
import pandas as pd
import json
import requests

Task 1:
- Prompt the user to input two or more stock symbols.
- Use the API to fetch data for the given symbols.
- Extract and display only the Stock Ticker, Company Name, and Current Market Price.

In [103]:
#example list of stocks
stock=input()
#ex: "AAPL,GOOGL,MSFT"

apikey="yourkey"

url = "https://yfapi.net/v6/finance/quote"

querystring = {"symbols":stock}

headers = {
    'x-api-key': apikey
    }
response = requests.request("GET", url, headers=headers, params=querystring)
df = pd.DataFrame(response.json()['quoteResponse']['result'])
df.head()

#converting to data frame
df = pd.DataFrame(response.json()['quoteResponse']['result'])

# Display only stock ticker, company name, and current market price
#print(df[['symbol', 'shortName','ask']])
for index, row in df.iterrows():
    print(f"Stock Ticker: {row['symbol']}, Company: {row['shortName']}, Current Market Price: ${row['ask']:.2f}")



AAPL
Stock Ticker: AAPL, Company: Apple Inc., Current Market Price: $240.34


Task 2.1:
- Allow the user to choose a module from the Quote Summary Endpoint.
- Find: 52 Week High, 52 Week Low, Return on Assets (ROA)
- Make an API request using the chosen module.
- Convert the response into a Pandas DataFrame.
- Display the DataFrame

In [104]:
modules = "summaryDetail,financialData"
#I am confused about the wording of this problem because I believe you need
#specific modules to find the information so I cannot allow user input
ticker = input()

url = f'https://yfapi.net/v11/finance/quoteSummary/{ticker}'

querystring = {
    "symbols": ticker,
    "modules": modules
}

headers = {
    'x-api-key': apikey
}

response = requests.request("GET", url, headers=headers, params=querystring)
#overall df
result = pd.DataFrame(response.json()["quoteSummary"]["result"][0])
#info from each needed module
summary_detail = result.get("summaryDetail")
fifty_two_week_high = summary_detail.get("fiftyTwoWeekHigh").get("raw", "N/A")
fifty_two_week_low = summary_detail.get("fiftyTwoWeekLow").get("raw", "N/A")
financial_data = result.get("financialData")
ROA = financial_data.get("returnOnAssets").get("raw", "N/A")
df2 = pd.DataFrame({
        "Stock Ticker": [ticker],
        "52-Week High": [fifty_two_week_high],
        "52-Week Low": [fifty_two_week_low],
        "ROA": [ROA]
    })

print(df2)


AAPL
  Stock Ticker  52-Week High  52-Week Low      ROA
0         AAPL         260.1       164.08  0.22519


Task 2.2:
- Find Current Trending Stocks , Their Actual Name and Ticker and Current Price as well as
their 52 high and low.

In [105]:
url = "https://yfapi.net/v1/finance/trending/US"  # Trending stocks in the US

headers = {'x-api-key': apikey}

response = requests.get(url, headers=headers)

#Trending stock tickers
trending_tickers = ",".join(
    stock["symbol"] for stock in response.json().get("finance").get("result", [{}])[0].get("quotes", [])
)
url = "https://yfapi.net/v6/finance/quote"

querystring = {
    "symbols": trending_tickers,
}

headers = {
    'x-api-key': '103ucaGMQ296wRcgi8mWE3lL6qNDywMh9QT3CCSG'
}

response = requests.request("GET", url, headers=headers, params=querystring)
df = pd.DataFrame(response.json()['quoteResponse']['result'])
df_filtered = df[['symbol', 'shortName', 'regularMarketPrice', 'fiftyTwoWeekHigh', 'fiftyTwoWeekLow']]
print(df_filtered)




   symbol                shortName  regularMarketPrice  fiftyTwoWeekHigh  \
0    NVDA       NVIDIA Corporation              131.28           153.130   
1    SNOW           Snowflake Inc.              166.19           233.880   
2     CRM         Salesforce, Inc.              307.33           369.000   
3    IONQ               IonQ, Inc.               29.93            54.740   
4    BYND        Beyond Meat, Inc.                3.56            12.120   
5    MRNA            Moderna, Inc.               33.58           170.470   
6      AI              C3.ai, Inc.               26.44            45.080   
7    TDOC     Teladoc Health, Inc.               10.99            15.950   
8    MARA      MARA Holdings, Inc.               12.45            34.090   
9     APP     Applovin Corporation              331.00           525.150   
10    NIO                 NIO Inc.                4.72             7.710   
11   EBAY                eBay Inc.               69.14            71.610   
12   NTNX   